# Detection + save output

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.predict(
    source="https://ultralytics.com/images/bus.jpg",
    save=True,
    conf=0.25,
    project="outputs",
    name="bus_demo",
)

print("Saved to:", results[0].save_dir)
print("Boxes:", results[0].boxes.shape)


# Instance segmentation

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n-seg.pt")

results = model.predict(
    source="https://ultralytics.com/images/bus.jpg",
    save=True,
    conf=0.25,
    project="outputs",
    name="bus_demo",
)

print("Saved to:", results[0].save_dir)
print("Boxes:", results[0].boxes.shape)


# Pose estimation

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n-pose.pt")

results = model.predict(
    source="https://ultralytics.com/images/bus.jpg",
    save=True,
    conf=0.25,
    project="outputs",
    name="bus_demo",
)

print("Saved to:", results[0].save_dir)
print("Boxes:", results[0].boxes.shape)


# YOLOE-26 supports both text-based and visual prompting

In [12]:
from ultralytics import YOLO

model = YOLO("yoloe-26l-seg.pt")

names = ["person", "bus"]
model.set_classes(names, model.get_text_pe(names))

results = model.predict(
    source="https://ultralytics.com/images/bus.jpg",
    save=True,
    conf=0.25,
    project="outputs",
    name="bus_demo",
)

print("Saved to:", results[0].save_dir)
print("Boxes:", results[0].boxes.shape)



Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
image 1/1 /home/logan/Desktop/YouTube-Videos/YOLO26-Introduction/YouTube-YOLO26-Introduction/bus.jpg: 640x480 5 persons, 1 bus, 10.1ms
Speed: 0.9ms preprocess, 10.1ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /home/logan/Desktop/YouTube-Videos/YOLO26-Introduction/YouTube-YOLO26-Introduction/outputs/bus_demo10
Saved to: /home/logan/Desktop/YouTube-Videos/YOLO26-Introduction/YouTube-YOLO26-Introduction/outputs/bus_demo10
Boxes: torch.Size([6, 6])


# Realtime Background Removal

In [2]:
import cv2
import numpy as np
from ultralytics import YOLO

model = YOLO("yolo26x-seg.pt")
bg = cv2.imread("bus.jpg")

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w = frame.shape[:2]
    bg_resized = cv2.resize(bg, (w, h))

    results = model(frame, conf=0.4, verbose=False)
    r = results[0]

    person_mask = np.zeros((h, w), dtype=np.uint8)

    if r.masks is not None:
        for i, cls in enumerate(r.boxes.cls):
            if int(cls) == 0:
                mask = r.masks.data[i].cpu().numpy()
                mask = cv2.resize(mask, (w, h))
                mask = (mask > 0.5).astype(np.uint8) * 255
                person_mask = cv2.bitwise_or(person_mask, mask)

    kernel = np.ones((5,5), np.uint8)
    person_mask = cv2.morphologyEx(person_mask, cv2.MORPH_CLOSE, kernel)
    person_mask = cv2.GaussianBlur(person_mask, (25,25), 0)

    person_mask_3 = cv2.cvtColor(person_mask, cv2.COLOR_GRAY2BGR) / 255.0

    foreground = frame * person_mask_3
    background = bg_resized * (1 - person_mask_3)
    final = (foreground + background).astype(np.uint8)

    cv2.imshow("YOLO Background Removed", final)

    key = cv2.waitKey(1)
    if key == 27:
        break


cap.release()
cv2.destroyAllWindows()
